In [ ]:
!pip install -q dbt-duckdb

In [ ]:
import os
try:
    import notebookutils
    vl             = notebookutils.variableLibrary.getLibrary("deploy_config")
    workspace_id   = vl.workspace_id
    download_limit = vl.download_limit
    process_limit  = vl.process_limit
    lakehouse_name = vl.lakehouse_name
    dbt_path       = vl.dbt_path
    metadata_path  = vl.metadata_path
    lakehouse_id   = notebookutils.lakehouse.get(lakehouse_name).get('id')
except ModuleNotFoundError:
    import yaml
    from pathlib import Path
    _root  = Path("C:/lakehouse/default/Files/dbt_fabric_python_notebook")
    _all   = yaml.safe_load((_root / "deploy_config.yml").read_text())
    _cfg   = {**_all.get("defaults", {}), **_all["local"]}
    workspace_id   = _cfg["ws_id"]
    lakehouse_id   = _cfg["lakehouse_id"]
    download_limit = _cfg["download_limit"]
    process_limit  = _cfg["process_limit"]
    dbt_path       = _cfg["dbt_path"]
    metadata_path  = _cfg["metadata_path"]
os.environ['ROOT_PATH']      = f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}'
os.environ['download_limit'] = download_limit
os.environ['process_limit']  = process_limit

In [ ]:
"""
OneLake metadata.db lock — protects DuckLake's SQLite catalog from
concurrent writers. See README "Concurrency safety" section for details.
"""
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone
import sqlite3

@contextmanager
def onelake_metadata_lock(metadata_path, workspace_id, lakehouse_id,
                          local_path="/tmp/metadata.db", stale_hours=12):
    """
    Hold an exclusive lease on metadata.db on OneLake while dbt runs locally.

    Flow:
      1. If the OneLake file is missing, seed it with a valid empty SQLite db.
      2. Acquire an infinite lease; auto-break if the prior lease is older
         than `stale_hours` (self-heal from a crashed run).
      3. Download to `local_path`, yield it.
      4. On exit: upload modifications back under the lease, then release.

    A second run hitting the same file fails fast with LeaseAlreadyPresent.
    """
    import notebookutils
    from azure.storage.filedatalake import DataLakeServiceClient, DataLakeLeaseClient
    from azure.core.credentials import AccessToken, TokenCredential
    from azure.core.exceptions import ResourceNotFoundError, HttpResponseError

    class _StaticToken(TokenCredential):
        def __init__(self, t): self.t = t
        def get_token(self, *_, **__): return AccessToken(self.t, 9999999999)

    # /lakehouse/default/Files/foo.db -> {lakehouse_id}/Files/foo.db
    marker = "/Files/"
    idx = metadata_path.find(marker)
    if idx < 0:
        raise ValueError(f"metadata_path missing '/Files/' segment: {metadata_path}")
    rel = f"{lakehouse_id}/Files/{metadata_path[idx + len(marker):]}"

    file = (
        DataLakeServiceClient(
            "https://onelake.dfs.fabric.microsoft.com",
            credential=_StaticToken(notebookutils.credentials.getToken("storage")),
        )
        .get_file_system_client(workspace_id)
        .get_file_client(rel)
    )

    # First-run seed: ADLS won't lease an unflushed zero-byte blob, and
    # DuckLake can't ATTACH an empty file. CREATE+DROP forces SQLite to
    # write a real header, giving us a leasable, attach-able placeholder.
    try:
        file.get_file_properties()
    except ResourceNotFoundError:
        seed = "/tmp/_metadata_seed.db"
        c = sqlite3.connect(seed)
        c.execute("CREATE TABLE _bootstrap (x)")
        c.execute("DROP TABLE _bootstrap")
        c.commit(); c.close()
        with open(seed, "rb") as fh:
            file.upload_data(fh.read(), overwrite=True)

    # Acquire — auto-heal a stale lease (>stale_hours) from a crashed run.
    try:
        lease = file.acquire_lease(lease_duration=-1)
    except HttpResponseError as e:
        if e.error_code != "LeaseAlreadyPresent":
            raise
        stamped = (file.get_file_properties().metadata or {}).get("acquired_at")
        is_stale = True
        if stamped:
            try:
                age = datetime.now(timezone.utc) - datetime.fromisoformat(stamped)
                is_stale = age > timedelta(hours=stale_hours)
            except ValueError:
                pass
        if not is_stale:
            raise RuntimeError(
                f"metadata.db is locked by an active writer (acquired_at={stamped}). "
                f"Refusing to run. Stale leases auto-break after {stale_hours}h."
            )
        DataLakeLeaseClient(file).break_lease(lease_break_period=0)
        lease = file.acquire_lease(lease_duration=-1)

    try:
        file.set_metadata(
            {"acquired_at": datetime.now(timezone.utc).isoformat(),
             "holder":      "fabric-notebook"},
            lease=lease.id,
        )
        with open(local_path, "wb") as fh:
            fh.write(file.download_file(lease=lease.id).readall())
        try:
            yield local_path
        finally:
            with open(local_path, "rb") as fh:
                file.upload_data(fh.read(), overwrite=True, lease=lease.id)
    finally:
        lease.release()

In [ ]:
from dbt.cli.main import dbtRunner

def run_dbt():
    os.chdir(dbt_path)
    dbt = dbtRunner()
    dbt.invoke(["run",  "--target", "prod", "--profiles-dir", "."])
    dbt.invoke(["test", "--target", "prod", "--profiles-dir", "."])

In [ ]:
from contextlib import nullcontext

try:
    import notebookutils  # noqa: F401
    lock = onelake_metadata_lock(metadata_path, workspace_id, lakehouse_id)
except ModuleNotFoundError:
    # Local dev: no Fabric, no lease — dbt opens the local mount directly.
    lock = nullcontext(metadata_path)

with lock as local_db:
    os.environ['METADATA_LOCAL_PATH'] = local_db
    run_dbt()